In [6]:
import sys
!"{sys.executable}" -m pip install xgboost
!"{sys.executable}" -m pip install optuna



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import optuna
from sklearn.metrics import mean_squared_error

In [8]:
data = np.load("../artifacts/dataset_sequences.npz")

X_train = data["X_train"]
X_val   = data["X_val"]
X_test  = data["X_test"]

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(216981, 24, 36)
(43963, 24, 36)
(28428, 24, 36)


In [9]:
pred = np.load("../artifacts/lstm_predictions.npz")

y_train_true = pred["y_train_true"]
y_val_true   = pred["y_val_true"]
y_test_true  = pred["y_test_true"]

y_train_pred = pred["y_train_pred"]
y_val_pred   = pred["y_val_pred"]
y_test_pred  = pred["y_test_pred"]

residual_train = pred["residual_train"]
residual_val   = pred["residual_val"]
residual_test  = pred["residual_test"]

In [10]:
# Ambil feature pada timestep terakhir dari setiap window
X_train_xgb = X_train[:, -1, :]
X_val_xgb   = X_val[:, -1, :]
X_test_xgb  = X_test[:, -1, :]

print("Shape sebelum tambah prediksi LSTM:")
print(X_train_xgb.shape)
print(X_val_xgb.shape)
print(X_test_xgb.shape)

Shape sebelum tambah prediksi LSTM:
(216981, 36)
(43963, 36)
(28428, 36)


In [11]:
# Tambahkan prediksi LSTM sebagai feature tambahan
X_train_xgb = np.hstack([
    X_train_xgb,
    y_train_pred.reshape(-1, 1)
])

X_val_xgb = np.hstack([
    X_val_xgb,
    y_val_pred.reshape(-1, 1)
])

X_test_xgb = np.hstack([
    X_test_xgb,
    y_test_pred.reshape(-1, 1)
])

print("Shape akhir:")
print(X_train_xgb.shape)
print(X_val_xgb.shape)
print(X_test_xgb.shape)

Shape akhir:
(216981, 37)
(43963, 37)
(28428, 37)


In [12]:
os.makedirs("../models", exist_ok=True)

In [13]:
xgb_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train_xgb,
    residual_train
)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,None


In [14]:
train_residual_pred = xgb_model.predict(X_train_xgb)
val_residual_pred = xgb_model.predict(X_val_xgb)
test_residual_pred = xgb_model.predict(X_test_xgb)

In [15]:
train_final = y_train_pred.ravel() + train_residual_pred
val_final = y_val_pred.ravel() + val_residual_pred
test_final = y_test_pred.ravel() + test_residual_pred

In [16]:
def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    nrmse = rmse / (np.max(y_true) - np.min(y_true))
    r2 = r2_score(y_true, y_pred)
    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "NRMSE": nrmse,
        "R2": r2
    }

In [17]:
train_metrics = evaluate(y_train_true, train_final)
val_metrics = evaluate(y_val_true, val_final)
test_metrics = evaluate(y_test_true, test_final)
results = pd.DataFrame(
    [train_metrics, val_metrics, test_metrics],
    index=["Train", "Validation", "Test"]
)
results

,MAE,MSE,RMSE,NRMSE,R2
Train,2.837082,31.673278,5.627902,0.005396,0.995736
Validation,2.604982,32.259837,5.679774,0.008619,0.994277
Test,2.431621,23.942780,4.893136,0.008350,0.987781


In [18]:
lstm_results = pd.read_csv(
    "../models/lstm_evaluation.csv",
    index_col=0
)

lstm_results

,MAE,MSE,RMSE,NRMSE,R²
Train,3.210650,50.317642,7.093493,0.006801,0.993226
Validation,2.725367,32.782827,5.725629,0.008688,0.994184
Test,2.504920,23.959700,4.894865,0.008353,0.987772


In [19]:
lstm_results = lstm_results.rename(columns={"R2": "R²"})
results = results.rename(columns={"R2": "R²"})

In [20]:
comparison = pd.DataFrame({
    "LSTM": lstm_results.loc["Test"],
    "LSTM + XGBoost": results.loc["Test"]
}).T

comparison

,MAE,MSE,RMSE,NRMSE,R²
LSTM,2.504920,23.95970,4.894865,0.008353,0.987772
LSTM + XGBoost,2.431621,23.94278,4.893136,0.008350,0.987781


In [21]:
mae_improvement = (
    (lstm_results.loc["Test", "MAE"] - results.loc["Test", "MAE"])
    / lstm_results.loc["Test", "MAE"]
) * 100

rmse_improvement = (
    (lstm_results.loc["Test", "RMSE"] - results.loc["Test", "RMSE"])
    / lstm_results.loc["Test", "RMSE"]
) * 100

r2_improvement = (
    (results.loc["Test", "R²"] - lstm_results.loc["Test", "R²"])
    / abs(lstm_results.loc["Test", "R²"])
) * 100

print(f"Improvement MAE  : {mae_improvement:.2f}%")
print(f"Improvement RMSE : {rmse_improvement:.2f}%")
print(f"Improvement R²   : {r2_improvement:.2f}%")

Improvement MAE  : 2.93%
Improvement RMSE : 0.04%
Improvement R²   : 0.00%


### Optuna Study

In [ ]:
def objective(trial):

    params = {

        "objective": "reg:squarederror",

        "n_estimators": trial.suggest_int(
            "n_estimators",
            100,
            1000
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.3,
            log=True
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            10
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.6,
            1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight",
            1,
            10
        ),

        "gamma": trial.suggest_float(
            "gamma",
            0,
            5
        ),

        "random_state": 42,

        "n_jobs": -1

    }

    model = XGBRegressor(**params)
    model.fit(

        X_train_xgb,

        residual_train

    )

    pred = model.predict(
        X_val_xgb
    )

    rmse = np.sqrt(

        mean_squared_error(

            residual_val,

            pred

        )

    )

    return rmse

In [23]:
def save_checkpoint(study, trial):

    joblib.dump(
        study,
        "../models/optuna_study.pkl"
    )

    study.trials_dataframe().to_csv(
        "../models/optuna_trials.csv",
        index=False
    )

In [24]:
study = optuna.create_study(
    direction="minimize"
)

study.optimize(
    objective,
    n_trials=50,
    callbacks=[save_checkpoint]
)

[I 2026-06-27 09:17:40,344] A new study created in memory with name: no-name-2b013ce5-5007-4dab-8b35-15a59abc2102
[I 2026-06-27 09:17:41,827] Trial 0 finished with value: 5.861307487340317 and parameters: {'n_estimators': 265, 'learning_rate': 0.24196639340396564, 'max_depth': 4, 'subsample': 0.9666175570087145, 'colsample_bytree': 0.7403442003328886, 'min_child_weight': 4, 'gamma': 1.8916321106892697}. Best is trial 0 with value: 5.861307487340317.
[I 2026-06-27 09:17:47,243] Trial 1 finished with value: 5.682330404831871 and parameters: {'n_estimators': 744, 'learning_rate': 0.015567351927421393, 'max_depth': 6, 'subsample': 0.8793363467109574, 'colsample_bytree': 0.6857822113414558, 'min_child_weight': 5, 'gamma': 3.1763991774148117}. Best is trial 1 with value: 5.682330404831871.
[I 2026-06-27 09:17:49,005] Trial 2 finished with value: 5.87490973256542 and parameters: {'n_estimators': 216, 'learning_rate': 0.2717196207631583, 'max_depth': 6, 'subsample': 0.7772777175718735, 'colsam

KeyboardInterrupt: 

In [26]:
print(study.best_params)

{'n_estimators': 856, 'learning_rate': 0.020148206559602352, 'max_depth': 10, 'subsample': 0.7475007635696106, 'colsample_bytree': 0.6456934819308223, 'min_child_weight': 1, 'gamma': 1.084835009652121}


In [25]:
best_xgb = XGBRegressor(
    **study.best_params,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

X_train_final = np.vstack([X_train_xgb, X_val_xgb])
y_train_final = np.concatenate([residual_train, residual_val])

best_xgb.fit(
    X_train_final,
    y_train_final
)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.6456934819308223
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,None


In [27]:
# Prediksi residual menggunakan model terbaik
train_residual_pred = best_xgb.predict(X_train_xgb)
val_residual_pred   = best_xgb.predict(X_val_xgb)
test_residual_pred  = best_xgb.predict(X_test_xgb)

In [28]:
train_final = y_train_pred.ravel() + train_residual_pred
val_final   = y_val_pred.ravel() + val_residual_pred
test_final  = y_test_pred.ravel() + test_residual_pred

In [29]:
train_metrics = evaluate(y_train_true, train_final)
val_metrics   = evaluate(y_val_true, val_final)
test_metrics  = evaluate(y_test_true, test_final)

tuned_results = pd.DataFrame(
    [train_metrics, val_metrics, test_metrics],
    index=["Train", "Validation", "Test"]
)

tuned_results

,MAE,MSE,RMSE,NRMSE,R2
Train,2.213165,15.759230,3.969790,0.003806,0.997878
Validation,1.923223,10.004096,3.162925,0.004800,0.998225
Test,2.387085,22.889397,4.784286,0.008164,0.988318


In [32]:
comparison = pd.DataFrame({
    "Baseline": results.loc["Test"],
    "Optuna": tuned_results.loc["Test"]
}).T

comparison

,MAE,MSE,NRMSE,R2,RMSE,R²
Baseline,2.431621,23.942780,0.008350,NaN,4.893136,0.987781
Optuna,2.387085,22.889397,0.008164,0.988318,4.784286,NaN


In [42]:
tuned_results = tuned_results.rename(columns={"R²": "R2"})
results = results.rename(columns={"R²": "R2"})

In [43]:
print(results.columns.tolist())
print(tuned_results.columns.tolist())

['MAE', 'MSE', 'RMSE', 'NRMSE', 'R2']
['MAE', 'MSE', 'RMSE', 'NRMSE', 'R2']


In [44]:
improvement = pd.Series({
    "MAE (%)": (
        (results.loc["Test", "MAE"] - tuned_results.loc["Test", "MAE"])
        / results.loc["Test", "MAE"]
    ) * 100,

    "RMSE (%)": (
        (results.loc["Test", "RMSE"] - tuned_results.loc["Test", "RMSE"])
        / results.loc["Test", "RMSE"]
    ) * 100,

    "R2 (%)": (
        (tuned_results.loc["Test", "R2"] - results.loc["Test", "R2"])
        / abs(results.loc["Test", "R2"])
    ) * 100
})

improvement

MAE (%)     1.831525
RMSE (%)    2.224535
R2 (%)      0.054424
dtype: float64

In [45]:
os.makedirs("../models", exist_ok=True)

joblib.dump(
    best_xgb,
    "../models/best_xgb_optuna.pkl"
)

print("Final XGBoost model berhasil disimpan.")

Final XGBoost model berhasil disimpan.


In [46]:
import json

with open("../models/best_xgb_params.json", "w") as f:
    json.dump(study.best_params, f, indent=4)

print("Best hyperparameters berhasil disimpan.")

Best hyperparameters berhasil disimpan.
